# Merkle-Hellman subset-sum demo

Small, reproducible walk-through for the toy cryptanalysis pipeline.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from annealing_crypto.experiments.run_benchmark import benchmark_config_from_mapping
from annealing_crypto.experiments.summarize_results import summarize_benchmark
from annealing_crypto.experiments.validate_results import validate_benchmark_csv
from annealing_crypto.metrics import hamming_distance
from annealing_crypto.merkle_hellman import decrypt_ciphertext, encrypt_bits, generate_keypair
from annealing_crypto.qubo import build_subset_sum_bqm
from annealing_crypto.solvers import (
    solve_brute_force,
    solve_exact_qubo,
    solve_simulated_annealing,
    solve_simulated_quantum_annealing,
)
from annealing_crypto.subset_sum import instance_from_merkle_hellman, objective_value

config_path = PROJECT_ROOT / "experiments/configs/report_benchmark.json"
benchmark_path = PROJECT_ROOT / "experiments/raw/benchmark.csv"
summary_path = PROJECT_ROOT / "experiments/processed/summary.csv"
plots_dir = PROJECT_ROOT / "experiments/plots"

pd.set_option("display.max_columns", 30)

## Reproducible experiment setup

The benchmark used for the report is described by a versioned JSON config. The generated CSV and plots are intentionally kept local, but the parameters below should make the run reproducible.

In [ ]:
benchmark_config = benchmark_config_from_mapping(json.loads(config_path.read_text()))
pd.DataFrame(
    {
        "parameter": benchmark_config.__dict__.keys(),
        "value": [repr(value) for value in benchmark_config.__dict__.values()],
    }
)

## Toy ciphertext

In [ ]:
keypair = generate_keypair(8, seed=42)
message = (1, 0, 1, 1, 0, 0, 1, 0)
ciphertext = encrypt_bits(keypair.public, message)
instance = instance_from_merkle_hellman(keypair.public, ciphertext, known_message=message)

{
    "public_weights": keypair.public.weights,
    "message": message,
    "ciphertext": ciphertext,
    "legal_decryption": decrypt_ciphertext(keypair.private, ciphertext),
}

## Solvers

In [ ]:
results = [
    solve_brute_force(instance),
    solve_exact_qubo(instance),
    solve_simulated_annealing(instance, num_reads=100, num_sweeps=500, seed=42),
    solve_simulated_quantum_annealing(
        instance,
        num_reads=20,
        num_sweeps=80,
        trotter_slices=4,
        seed=42,
    ),
]

pd.DataFrame(
    {
        "solver": result.solver,
        "solution": result.solution,
        "objective_value": objective_value(instance, result.solution),
        "exact_hit": result.exact_hit,
        "matches_message": result.solution == message,
        "hamming_distance": hamming_distance(result.solution, message),
        "runtime_ms": result.runtime_ms,
    }
    for result in results
)

## QUBO energy

In [ ]:
bqm = build_subset_sum_bqm(instance)
{
    "variables": len(bqm.variables),
    "quadratic_terms": len(bqm.quadratic),
    "energy_for_message": bqm.energy(dict(enumerate(message))),
}

## Benchmark CSV

`exact_hit_rate` means the solver found a subset with objective value 0. `known_solution_match_rate` is stricter: it checks whether the recovered bits match the planted message/vector.

In [ ]:
if benchmark_path.exists():
    validation_report = validate_benchmark_csv(benchmark_path, config=benchmark_config)
    display(validation_report.as_dict())
    benchmark = pd.read_csv(benchmark_path)
    display(benchmark.head())
    if summary_path.exists():
        display(pd.read_csv(summary_path))
    else:
        display(summarize_benchmark(benchmark_path))
else:
    print("Run annealing-crypto-benchmark first to create experiments/raw/benchmark.csv")

### Short interpretation

- `Brute force` is the correctness baseline: it should hit exact solutions for these small instances, but its cost grows exponentially.
- `Simulated annealing` and the local `simulated quantum annealing` solver are heuristics. Their hit rate drops as the Merkle-Hellman-derived instances grow.
- The SQA solver here is a local simulation with Trotter replicas, not a physical QPU run. Its runtime is therefore CPU/Python overhead plus the simulated annealing schedule.
- `exact_hit_rate` is the main subset-sum success metric. `known_solution_match_rate` is stricter and matters when interpreting recovery of the planted message bits.

## Generated plots

In [ ]:
plot_names = [
    "success_rate_vs_n.png",
    "runtime_ms_vs_n.png",
    "objective_value_vs_n.png",
    "hamming_distance_vs_n.png",
]

for plot_name in plot_names:
    plot_path = plots_dir / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path)))
    else:
        print(f"Missing plot: {plot_path}")